# DINOv2-Small U2 online self-training with EMA teacher + ClassMix

This notebook upgrades `U1` into an online UDA loop:
- paired `Cityscapes` source + `ACDC train` target batches,
- on-the-fly pseudo-labels from an `EMA` teacher,
- dynamic confidence threshold from `0.968 -> 0.90`,
- `ClassMix` source-to-target compositing,
- supervised validation on `Cityscapes val`,
- checkpoint export for later `ACDC` evaluation.


In [1]:
!pip install -q torchmetrics pytorch-lightning clearml python-dotenv torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.9 MB/s eta 0:00:00


In [2]:
import copy
import math
import os
import shutil
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback, TQDMProgressBar
from torchmetrics import JaccardIndex

from clearml import Task
from google.colab import drive, userdata


In [3]:
CONFIG = {
    "project_name": "Segmentation_Urban_Scene_CourseWork",
    "task_name": "DinoV2_Small_U2_Cityscapes_ACDC_EMAClassMix",
    "cityscapes_dir": "/content/data/cityscapes",
    "cityscapes_zips": [
        "leftImg8bit_trainvaltest.zip",
        "gtFine_trainvaltest.zip",
    ],
    "drive_root": "/content/drive/MyDrive",
    "init_weights_rel_path": "weights/dinov2-small-U1-cityscapes-acdc-*.ckpt",
    "fallback_init_weights_rel_path": "weights/dinov2-small-E3-cityscapes-epoch=44-val_miou=0.6406.ckpt",
    "acdc_zips": [
        "rgb_anon_trainvaltest.zip",
    ],
    "acdc_data_dir": "/content/data/acdc",
    "weights_export_rel_dir": "weights",
    "conditions": ["fog", "night", "rain", "snow"],
    "model_name": "dinov2_vits14",
    "classes": 19,
    "batch_size": 8,
    "lr": 3e-5,
    "weight_decay": 0.05,
    "epochs": 15,
    "image_size": (518, 1022),
    "freeze_backbone": False,
    "num_workers": 4,
    "source_loss_weight": 1.0,
    "target_loss_weight": 0.75,
    "ema_alpha": 0.999,
    "initial_threshold": 0.968,
    "final_threshold": 0.90,
    "classmix_prob": 0.5,
    "strong_aug": True,
    "check_val_every_n_epoch": 1,
    "early_stopping_patience": 4,
    "early_stopping_min_delta": 0.001,
    "accumulate_grad_batches": 4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

torch.set_float32_matmul_precision("medium")
pl.seed_everything(42, workers=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["CLEARML_API_ACCESS_KEY"] = userdata.get("CLEARML_API_ACCESS_KEY")
os.environ["CLEARML_API_SECRET_KEY"] = userdata.get("CLEARML_API_SECRET_KEY")

task = Task.init(
    project_name=CONFIG["project_name"],
    task_name=CONFIG["task_name"],
    output_uri=False,
)
task.connect(CONFIG)

CONFIG


INFO:lightning_fabric.utilities.seed:Seed set to 42


ClearML Task: created new task id=850f670690f14305b8463d502c24c0b9


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


2026-03-29 17:19:19,880 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/588d5d925230490b9f98b1007b4c7fa0/experiments/850f670690f14305b8463d502c24c0b9/output/log


{'project_name': 'Segmentation_Urban_Scene_CourseWork',
 'task_name': 'DinoV2_Small_U2_Cityscapes_ACDC_EMAClassMix',
 'cityscapes_dir': '/content/data/cityscapes',
 'cityscapes_zips': ['leftImg8bit_trainvaltest.zip',
  'gtFine_trainvaltest.zip'],
 'drive_root': '/content/drive/MyDrive',
 'init_weights_rel_path': 'weights/dinov2-small-U1-cityscapes-acdc-*.ckpt',
 'fallback_init_weights_rel_path': 'weights/dinov2-small-E3-cityscapes-epoch=44-val_miou=0.6406.ckpt',
 'acdc_zips': ['rgb_anon_trainvaltest.zip'],
 'acdc_data_dir': '/content/data/acdc',
 'weights_export_rel_dir': 'weights',
 'conditions': ['fog', 'night', 'rain', 'snow'],
 'model_name': 'dinov2_vits14',
 'classes': 19,
 'batch_size': 8,
 'lr': 3e-05,
 'weight_decay': 0.05,
 'epochs': 15,
 'image_size': (518, 1022),
 'freeze_backbone': False,
 'num_workers': 4,
 'source_loss_weight': 1.0,
 'target_loss_weight': 0.75,
 'ema_alpha': 0.999,
 'initial_threshold': 0.968,
 'final_threshold': 0.9,
 'classmix_prob': 0.5,
 'strong_aug':

In [4]:
drive.mount("/content/drive")

Path(CONFIG["cityscapes_dir"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["acdc_data_dir"]).mkdir(parents=True, exist_ok=True)

for zip_name in CONFIG["cityscapes_zips"]:
    zip_path = Path(CONFIG["drive_root"]) / zip_name
    if not zip_path.exists():
        raise FileNotFoundError(f"Cityscapes zip not found: {zip_path}")

    print(f"Unpacking {zip_path} -> {CONFIG['cityscapes_dir']}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(CONFIG["cityscapes_dir"])

for zip_name in CONFIG["acdc_zips"]:
    zip_path = Path(CONFIG["drive_root"]) / zip_name
    if not zip_path.exists():
        raise FileNotFoundError(f"ACDC zip not found: {zip_path}")

    print(f"Unpacking {zip_path} -> {CONFIG['acdc_data_dir']}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(CONFIG["acdc_data_dir"])


def resolve_checkpoint(root_dir, pattern):
    if any(char in pattern for char in "*?["):
        candidates = sorted(root_dir.glob(pattern), key=lambda p: p.stat().st_mtime)
        if not candidates:
            raise FileNotFoundError(f"No checkpoint matched: {root_dir / pattern}")
        return candidates[-1]
    checkpoint_path = root_dir / pattern
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    return checkpoint_path


cityscapes_left = Path(CONFIG["cityscapes_dir"]) / "leftImg8bit" / "train"
cityscapes_gt = Path(CONFIG["cityscapes_dir"]) / "gtFine" / "train"
acdc_train_root = Path(CONFIG["acdc_data_dir"]) / "rgb_anon"
if not cityscapes_left.exists() or not cityscapes_gt.exists():
    raise FileNotFoundError(
        f"Cityscapes train folders not found after extraction: {cityscapes_left} | {cityscapes_gt}"
    )
if not acdc_train_root.exists():
    raise FileNotFoundError(f"ACDC rgb_anon root not found after extraction: {acdc_train_root}")

weights_root = Path(CONFIG["drive_root"])
try:
    init_weights_path = resolve_checkpoint(weights_root, CONFIG["init_weights_rel_path"])
except FileNotFoundError:
    init_weights_path = resolve_checkpoint(weights_root, CONFIG["fallback_init_weights_rel_path"])

export_weights_dir = weights_root / CONFIG["weights_export_rel_dir"]
export_weights_dir.mkdir(parents=True, exist_ok=True)

print(f"Cityscapes dir: {CONFIG['cityscapes_dir']}")
print(f"ACDC dir: {CONFIG['acdc_data_dir']}")
print(f"Warm-start checkpoint: {init_weights_path}")


Mounted at /content/drive
Unpacking /content/drive/MyDrive/leftImg8bit_trainvaltest.zip -> /content/data/cityscapes
Unpacking /content/drive/MyDrive/gtFine_trainvaltest.zip -> /content/data/cityscapes
Unpacking /content/drive/MyDrive/rgb_anon_trainvaltest.zip -> /content/data/acdc
Cityscapes dir: /content/data/cityscapes
ACDC dir: /content/data/acdc
Warm-start checkpoint: /content/drive/MyDrive/weights/dinov2-small-U1-cityscapes-acdc-epoch=05-val_miou=0.7110.ckpt


In [5]:
CITYSCAPES_ID_TO_TRAIN_ID = {
    7: 0,
    8: 1,
    11: 2,
    12: 3,
    13: 4,
    17: 5,
    19: 6,
    20: 7,
    21: 8,
    22: 9,
    23: 10,
    24: 11,
    25: 12,
    26: 13,
    27: 14,
    28: 15,
    31: 16,
    32: 17,
    33: 18,
}


class CityscapesDataset(Dataset):
    def __init__(self, root_dir, split="train", image_size=(518, 1022), augment=False):
        self.root_dir = Path(root_dir)
        self.split = split
        self.image_size = image_size
        self.augment = augment
        self.items = []

        images_dir = self.root_dir / "leftImg8bit" / split
        masks_dir = self.root_dir / "gtFine" / split
        for city_dir in sorted(images_dir.iterdir()):
            if not city_dir.is_dir():
                continue
            mask_dir = masks_dir / city_dir.name
            for image_path in sorted(city_dir.glob("*_leftImg8bit.png")):
                mask_path = mask_dir / image_path.name.replace("_leftImg8bit.png", "_gtFine_labelIds.png")
                if mask_path.exists():
                    self.items.append({"image": image_path, "mask": mask_path})

        if len(self.items) == 0:
            raise RuntimeError(f"No Cityscapes pairs found in: {images_dir}")

    def __len__(self):
        return len(self.items)

    def _resize(self, image, mask):
        height, width = self.image_size
        image = cv2.resize(image, (width, height), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
        return image, mask

    def _map_mask(self, mask):
        mapped = np.full(mask.shape, 255, dtype=np.uint8)
        for raw_id, train_id in CITYSCAPES_ID_TO_TRAIN_ID.items():
            mapped[mask == raw_id] = train_id
        return mapped

    def __getitem__(self, idx):
        sample = self.items[idx]
        image = cv2.imread(str(sample["image"]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(sample["mask"]), 0)

        image, mask = self._resize(image, mask)
        mask = self._map_mask(mask)

        if self.augment and torch.rand(1).item() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])

        image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        mask = torch.from_numpy(mask.astype(np.int64))
        return {"image": image, "mask": mask}


class ACDCUnlabeledDataset(Dataset):
    def __init__(self, root_dir, split="train", conditions=None, image_size=(518, 1022), augment=False):
        self.root_dir = Path(root_dir)
        self.split = split
        self.conditions = conditions or ["fog", "night", "rain", "snow"]
        self.image_size = image_size
        self.augment = augment
        self.items = []

        for condition in self.conditions:
            rgb_root = self.root_dir / "rgb_anon" / condition / split
            if not rgb_root.exists():
                continue

            for image_path in sorted(rgb_root.rglob("*_rgb_anon.png")):
                self.items.append({
                    "image": image_path,
                    "condition": condition,
                    "stem": image_path.stem,
                })

        if len(self.items) == 0:
            raise RuntimeError("No ACDC target images found.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        sample = self.items[idx]
        image = cv2.imread(str(sample["image"]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        height, width = self.image_size
        image = cv2.resize(image, (width, height), interpolation=cv2.INTER_LINEAR)

        if self.augment and torch.rand(1).item() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])

        image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        return {"image": image}


class PairedUDADataset(Dataset):
    def __init__(self, source_dataset, target_dataset):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset

    def __len__(self):
        return len(self.source_dataset)

    def __getitem__(self, idx):
        source = self.source_dataset[idx]
        target_idx = torch.randint(len(self.target_dataset), size=(1,)).item()
        target = self.target_dataset[target_idx]
        return {
            "source_image": source["image"],
            "source_mask": source["mask"],
            "target_image": target["image"],
        }


class RoadDataModule(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        self.config = config

    def setup(self, stage=None):
        self.source_train_ds = CityscapesDataset(
            self.config["cityscapes_dir"],
            split="train",
            image_size=self.config["image_size"],
            augment=True,
        )
        self.target_train_ds = ACDCUnlabeledDataset(
            self.config["acdc_data_dir"],
            split="train",
            conditions=self.config["conditions"],
            image_size=self.config["image_size"],
            augment=True,
        )
        self.train_ds = PairedUDADataset(self.source_train_ds, self.target_train_ds)
        self.val_ds = CityscapesDataset(
            self.config["cityscapes_dir"],
            split="val",
            image_size=self.config["image_size"],
            augment=False,
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.config["batch_size"],
            shuffle=True,
            num_workers=self.config["num_workers"],
            pin_memory=True,
            persistent_workers=self.config["num_workers"] > 0,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.config["batch_size"],
            shuffle=False,
            num_workers=self.config["num_workers"],
            pin_memory=True,
            persistent_workers=self.config["num_workers"] > 0,
        )


In [6]:
class LinearSegmentationHead(nn.Module):
    def __init__(self, embed_dim=384, num_classes=19, patch_size=14):
        super().__init__()
        self.patch_size = patch_size
        self.linear = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        batch_size, _, channels = x.shape
        patch_h = h // self.patch_size
        patch_w = w // self.patch_size
        x = x.reshape(batch_size, patch_h, patch_w, channels).permute(0, 3, 1, 2)
        x = self.linear(x)
        x = F.interpolate(x, size=(h, w), mode="bilinear", align_corners=False)
        return x


class ClearMLMetricsCallback(Callback):
    def __init__(self, task):
        super().__init__()
        self._logger = task.get_logger()

    def _log_metrics(self, trainer, stage):
        for name, value in trainer.callback_metrics.items():
            if isinstance(value, torch.Tensor):
                value = value.detach().cpu().item()
            if isinstance(value, (int, float)) and math.isfinite(value):
                self._logger.report_scalar(stage, name, value, iteration=trainer.current_epoch)

    def on_train_epoch_end(self, trainer, pl_module):
        self._log_metrics(trainer, "train")

    def on_validation_epoch_end(self, trainer, pl_module):
        self._log_metrics(trainer, "val")


class RoadModel(pl.LightningModule):
    def __init__(
        self,
        model_name="dinov2_vits14",
        num_classes=19,
        lr=3e-5,
        weight_decay=0.05,
        epochs=15,
        freeze_backbone=False,
        source_loss_weight=1.0,
        target_loss_weight=0.75,
        ema_alpha=0.999,
        initial_threshold=0.968,
        final_threshold=0.90,
        classmix_prob=0.5,
        strong_aug=True,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.backbone = torch.hub.load("facebookresearch/dinov2", model_name)
        self.embed_dim = 384
        self.patch_size = 14
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.segmentation_head = LinearSegmentationHead(
            embed_dim=self.embed_dim,
            num_classes=num_classes,
            patch_size=self.patch_size,
        )

        self.teacher_backbone = None
        self.teacher_segmentation_head = None

        self.train_source_miou = JaccardIndex(task="multiclass", num_classes=num_classes, ignore_index=255)
        self.val_miou = JaccardIndex(task="multiclass", num_classes=num_classes, ignore_index=255)

    def _normalize(self, images):
        mean = images.new_tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
        std = images.new_tensor(IMAGENET_STD).view(1, 3, 1, 1)
        return (images - mean) / std

    def _forward_with_modules(self, backbone, segmentation_head, images):
        _, _, h, w = images.shape
        features = backbone.forward_features(self._normalize(images))
        patch_features = features["x_norm_patchtokens"]
        return segmentation_head(patch_features, h, w)

    def forward(self, images):
        return self._forward_with_modules(self.backbone, self.segmentation_head, images)

    def initialize_teacher_from_student(self):
        self.teacher_backbone = copy.deepcopy(self.backbone)
        self.teacher_segmentation_head = copy.deepcopy(self.segmentation_head)
        for module in [self.teacher_backbone, self.teacher_segmentation_head]:
            module.eval()
            for param in module.parameters():
                param.requires_grad = False

    @torch.no_grad()
    def _teacher_forward(self, images):
        if self.teacher_backbone is None or self.teacher_segmentation_head is None:
            self.initialize_teacher_from_student()
        return self._forward_with_modules(self.teacher_backbone, self.teacher_segmentation_head, images)

    @torch.no_grad()
    def _update_teacher(self):
        if self.teacher_backbone is None or self.teacher_segmentation_head is None:
            self.initialize_teacher_from_student()

        alpha = self.hparams.ema_alpha
        for teacher_param, student_param in zip(self.teacher_backbone.parameters(), self.backbone.parameters()):
            teacher_param.data.mul_(alpha).add_(student_param.data, alpha=1.0 - alpha)
        for teacher_param, student_param in zip(self.teacher_segmentation_head.parameters(), self.segmentation_head.parameters()):
            teacher_param.data.mul_(alpha).add_(student_param.data, alpha=1.0 - alpha)

        for teacher_buffer, student_buffer in zip(self.teacher_backbone.buffers(), self.backbone.buffers()):
            teacher_buffer.data.copy_(student_buffer.data)
        for teacher_buffer, student_buffer in zip(self.teacher_segmentation_head.buffers(), self.segmentation_head.buffers()):
            teacher_buffer.data.copy_(student_buffer.data)

        self.teacher_backbone.eval()
        self.teacher_segmentation_head.eval()

    def on_fit_start(self):
        if self.teacher_backbone is None or self.teacher_segmentation_head is None:
            self.initialize_teacher_from_student()

    def on_save_checkpoint(self, checkpoint):
        teacher_keys = [
            key
            for key in list(checkpoint["state_dict"].keys())
            if key.startswith("teacher_backbone.") or key.startswith("teacher_segmentation_head.")
        ]
        for key in teacher_keys:
            del checkpoint["state_dict"][key]

    def _masked_ce(self, logits, masks):
        loss_map = F.cross_entropy(logits, masks, ignore_index=255, reduction="none")
        valid = (masks != 255).float()
        denom = valid.sum().clamp_min(1.0)
        return (loss_map * valid).sum() / denom

    def _current_threshold(self):
        if self.hparams.epochs <= 1:
            return float(self.hparams.final_threshold)
        progress = min(1.0, self.current_epoch / max(self.hparams.epochs - 1, 1))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return float(
            self.hparams.final_threshold
            + (self.hparams.initial_threshold - self.hparams.final_threshold) * cosine
        )

    def _sample_classmix_masks(self, source_masks):
        mix_masks = []
        selected_fracs = []
        for mask in source_masks:
            valid_classes = torch.unique(mask[mask != 255])
            if len(valid_classes) == 0:
                mix_masks.append(torch.zeros_like(mask, dtype=torch.bool))
                selected_fracs.append(0.0)
                continue

            num_select = max(1, int(math.ceil(len(valid_classes) * 0.5)))
            perm = torch.randperm(len(valid_classes), device=mask.device)
            selected = valid_classes[perm[:num_select]]

            sample_mask = torch.zeros_like(mask, dtype=torch.bool)
            for cls_id in selected:
                sample_mask |= mask == cls_id

            mix_masks.append(sample_mask)
            selected_fracs.append(num_select / float(len(valid_classes)))

        mean_selected = float(np.mean(selected_fracs)) if selected_fracs else 0.0
        return torch.stack(mix_masks, dim=0), mean_selected

    def _gaussian_blur(self, images, kernel_size=5, sigma=1.0):
        coords = torch.arange(kernel_size, device=images.device, dtype=images.dtype)
        coords = coords - kernel_size // 2
        kernel_1d = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        kernel_1d = kernel_1d / kernel_1d.sum()
        kernel_2d = torch.outer(kernel_1d, kernel_1d)
        kernel = kernel_2d.expand(images.shape[1], 1, kernel_size, kernel_size)
        return F.conv2d(images, kernel, padding=kernel_size // 2, groups=images.shape[1])

    def _apply_strong_aug(self, images):
        if not self.hparams.strong_aug:
            return images

        out = images.clone()
        if torch.rand(1).item() < 0.8:
            brightness = 1.0 + (torch.rand(out.shape[0], 1, 1, 1, device=out.device) - 0.5) * 0.4
            contrast = 1.0 + (torch.rand(out.shape[0], 1, 1, 1, device=out.device) - 0.5) * 0.4
            mean = out.mean(dim=(2, 3), keepdim=True)
            out = (out - mean) * contrast + mean
            out = out * brightness

        if torch.rand(1).item() < 0.3:
            sigma = float(torch.empty(1, device=out.device).uniform_(0.5, 1.5).item())
            out = self._gaussian_blur(out, kernel_size=5, sigma=sigma)

        return out.clamp(0.0, 1.0)

    def training_step(self, batch, batch_idx):
        source_images = batch["source_image"].float()
        source_masks = batch["source_mask"].long()
        target_images = batch["target_image"].float()

        source_logits = self(source_images)
        source_loss = self._masked_ce(source_logits, source_masks)

        threshold = self._current_threshold()
        with torch.no_grad():
            teacher_logits = self._teacher_forward(target_images)
            teacher_probs = torch.softmax(teacher_logits, dim=1)
            pseudo_confidence, pseudo_labels = torch.max(teacher_probs, dim=1)

        confidence_mask = pseudo_confidence >= threshold
        mixed_images = target_images.clone()
        mixed_labels = pseudo_labels.clone()
        mixed_labels[~confidence_mask] = 255

        classmix_applied = float(torch.rand(1).item() < self.hparams.classmix_prob)
        classmix_ratio = torch.tensor(0.0, device=self.device)
        if classmix_applied > 0.0:
            classmix_masks, selected_fraction = self._sample_classmix_masks(source_masks)
            classmix_ratio = torch.tensor(selected_fraction, device=self.device)
            mixed_images = torch.where(classmix_masks.unsqueeze(1), source_images, mixed_images)
            mixed_labels = torch.where(classmix_masks, source_masks, mixed_labels)

        mixed_inputs = self._apply_strong_aug(mixed_images)
        mixed_logits = self(mixed_inputs)
        mixed_loss = self._masked_ce(mixed_logits, mixed_labels)

        loss = (
            self.hparams.source_loss_weight * source_loss
            + self.hparams.target_loss_weight * mixed_loss
        )

        source_preds = torch.argmax(source_logits, dim=1)
        self.train_source_miou(source_preds, source_masks)

        mixed_keep_ratio = (mixed_labels != 255).float().mean()
        teacher_confidence_mean = pseudo_confidence.mean()

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_source_loss", source_loss, on_step=False, on_epoch=True, logger=True)
        self.log("train_mixed_loss", mixed_loss, on_step=False, on_epoch=True, logger=True)
        self.log("train_source_miou", self.train_source_miou, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_threshold", threshold, on_step=False, on_epoch=True, logger=True)
        self.log("train_pseudo_keep_ratio", mixed_keep_ratio, on_step=False, on_epoch=True, logger=True)
        self.log("train_teacher_confidence", teacher_confidence_mean, on_step=False, on_epoch=True, logger=True)
        self.log("train_classmix_applied", classmix_applied, on_step=False, on_epoch=True, logger=True)
        self.log("train_classmix_ratio", classmix_ratio, on_step=False, on_epoch=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images = batch["image"].float()
        masks = batch["mask"].long()
        logits = self(images)
        loss = self._masked_ce(logits, masks)
        preds = torch.argmax(logits, dim=1)
        self.val_miou(preds, masks)

        self.log("val_loss", loss, on_epoch=True, prog_bar=True, logger=True)
        self.log("val_miou", self.val_miou, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def configure_optimizers(self):
        if self.hparams.freeze_backbone:
            params = self.segmentation_head.parameters()
        else:
            params = [
                {"params": self.segmentation_head.parameters(), "lr": self.hparams.lr},
                {"params": self.backbone.parameters(), "lr": self.hparams.lr * 0.1},
            ]

        optimizer = torch.optim.AdamW(
            params,
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
            betas=(0.9, 0.999),
            eps=1e-8,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.hparams.epochs,
            eta_min=1e-6,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
            },
        }

    def optimizer_step(self, epoch, batch_idx, optimizer, optimizer_closure):
        optimizer.step(closure=optimizer_closure)
        self._update_teacher()


def load_lightning_weights_into_model(model, ckpt_path, device):
    checkpoint = torch.load(ckpt_path, map_location=device)
    state_dict = checkpoint.get("state_dict", checkpoint)

    filtered_state = {}
    for key, value in state_dict.items():
        if key.startswith("backbone.") or key.startswith("segmentation_head."):
            filtered_state[key] = value

    missing, unexpected = model.load_state_dict(filtered_state, strict=False)
    print(f"Loaded init checkpoint from: {ckpt_path}")
    print(f"Missing keys: {len(missing)}")
    print(f"Unexpected keys: {len(unexpected)}")


In [7]:
dm = RoadDataModule(CONFIG)
dm.setup()

print(f"Cityscapes train samples: {len(dm.source_train_ds)}")
print(f"ACDC unlabeled train samples: {len(dm.target_train_ds)}")
print(f"Paired train samples per epoch: {len(dm.train_ds)}")

model = RoadModel(
    model_name=CONFIG["model_name"],
    num_classes=CONFIG["classes"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    epochs=CONFIG["epochs"],
    freeze_backbone=CONFIG["freeze_backbone"],
    source_loss_weight=CONFIG["source_loss_weight"],
    target_loss_weight=CONFIG["target_loss_weight"],
    ema_alpha=CONFIG["ema_alpha"],
    initial_threshold=CONFIG["initial_threshold"],
    final_threshold=CONFIG["final_threshold"],
    classmix_prob=CONFIG["classmix_prob"],
    strong_aug=CONFIG["strong_aug"],
)
load_lightning_weights_into_model(model, str(init_weights_path), CONFIG["device"])

checkpoint_callback = pl.callbacks.ModelCheckpoint(
    monitor="val_miou",
    dirpath="./checkpoints",
    filename="dinov2-small-U2-cityscapes-acdc-{epoch:02d}-{val_miou:.4f}",
    save_top_k=1,
    mode="max",
)

early_stopping = pl.callbacks.EarlyStopping(
    monitor="val_miou",
    patience=CONFIG["early_stopping_patience"],
    min_delta=CONFIG["early_stopping_min_delta"],
    mode="max",
)

trainer = pl.Trainer(
    max_epochs=CONFIG["epochs"],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    logger=True,
    callbacks=[
        checkpoint_callback,
        early_stopping,
        TQDMProgressBar(refresh_rate=20),
        ClearMLMetricsCallback(task),
    ],
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    accumulate_grad_batches=CONFIG["accumulate_grad_batches"],
    check_val_every_n_epoch=CONFIG["check_val_every_n_epoch"],
)

trainer.fit(model, datamodule=dm)


Cityscapes train samples: 2975
ACDC unlabeled train samples: 1600
Paired train samples per epoch: 2975
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning:

xFormers is not available (SwiGLU)

/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning:

xFormers is not available (Attention)

/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning:

xFormers is not available (Block)



Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 221MB/s]


2026-03-29 17:29:05,218 - clearml.model - INFO - Selected model id: 096af8490ce44af1b6986ef1b35ef075
2026-03-29 17:29:13,479 - clearml.model - INFO - Selected model id: 3223a17fa1de490db05aa030355eec05


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [8]:
best_model_path = checkpoint_callback.best_model_path
if not best_model_path:
    raise RuntimeError("Training finished without a saved best checkpoint.")

best_model_path = Path(best_model_path)
export_path = export_weights_dir / best_model_path.name
shutil.copy2(best_model_path, export_path)
print(f"Copied best checkpoint to: {export_path}")

best_model = RoadModel.load_from_checkpoint(str(best_model_path), map_location=CONFIG["device"])
best_model = best_model.to(CONFIG["device"])
val_metrics = trainer.validate(best_model, datamodule=dm)
display(pd.DataFrame(val_metrics))


Copied best checkpoint to: /content/drive/MyDrive/weights/dinov2-small-U2-cityscapes-acdc-epoch=01-val_miou=0.7141.ckpt


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


2026-03-29 17:59:47,591 - clearml.model - WARNING - Connecting multiple input models with the same name: `dinov2_vits14_pretrain`. This might result in the wrong model being used when executing remotely


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val_loss          │    0.1613599807024002     │
│         val_miou          │    0.7140748500823975     │
└───────────────────────────┴───────────────────────────┘

,val_loss,val_miou
0,0.16136,0.714075
